**FEATURE SELECTION CODE**

In [0]:
# Databricks only. Run this once if packages are missing.
%pip install -U lightgbm pyarrow pandas numpy scikit-learn psutil

# After install in Databricks:
dbutils.library.restartPython()

In [0]:
import os
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import lightgbm as lgb
import psutil

from sklearn.metrics import r2_score

In [0]:
# ============================================================
# Input / output config
# ============================================================

from pathlib import Path

import numpy as np
import psutil
import pyarrow.dataset as ds

# ============================================================
# Paths
# ============================================================

DATA_PATH = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

OUTPUT_DIR = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/v2/"
)

dbutils.fs.mkdirs(f"dbfs:{OUTPUT_DIR}")

# ============================================================
# Column config
# ============================================================

TARGET_COL = "TOTAL_NBRX_C4"

EXCLUDE_COLS = {
    TARGET_COL,
    "PROVIDER_ID", "E_FIELD_ADDED_NOT_SEGMENTED","D_LOWER_POTENTIAL_NON_WRITER_SEGMENT","UNASSIGNED_SEGMENT",'ADVISORY_BOARD_ATTENDED_R_P3', 'DOCEREE_BANNER_IMPRESSION_R_P3', 'DOXIMITY_FORMULARYONE_VIEW_R_P3', 'EPOCRATES_DOCALERT_DOCALERT_DELIVERED_R_P3', 'EPOCRATES_MONOGRAPH_MESSAGE_MONOGRAPH_MESSAGE_HEADLINE_IMPRESSION_R_P3', 'EPOCRATES_SEARCH_SUGGESTIONS_SEARCH_SUGGESTION_HEADLINE_IMPRESSION_R_P3', 'IMPIRICUS_MOBILE_SEND_R_P3', 'INVESTIGATOR_MEETING_ATTENDED_R_P3', 'MARKET_ACCESS_FORMULARY_ASSIST_ATTENDED_R_P3', 'MD_CALC_BRANDALERT_ENGAGEMENT_R_P3', 'MEDSCAPE_BRANDSPOT_RICH_MEDIA_ENGAGEMENT_R_P3', 'MEDSCAPE_EHR_EXPOSURE_R_P3', 'MEDSCAPE_FFQ_PAGE_VIEW_R_P3', 'MEDSCAPE_RECAP_PAGE_VIEW_R_P3', 'MEDSCAPE_XCOD_VIEW_R_P3', 'MJHLIFESCIENCES_3RD_PARTY_EMAIL_SENT_R_P3', 'MJHLIFESCIENCES_BANNER_IMPRESSIONS_R_P3', 'NATIVO_BANNER_IMPRESSIONS_R_P3', 'PEERDIRECT_FFQ_VIEW_R_P3', 'PEERDIRECT_MDFI_WEB_VIEW_R_P3', 'PRESCRIBER_POINT_CUSTOM_IMPRESSION_R_P3', 'REACHMD_DESKTOP_PAGE_VIEW_R_P3', 'REACHMD_EMAIL_EMAIL_DELIVERED_R_P3', 'SCIENTIFIC_PRESENTATION_ATTENDED_R_P3', 'SCIENTIFIC_ROUND_TABLES_ATTENDED_R_P3', 'SEARCH_BANNER_CLICKED_R_P3', 'SPONSORSHIP_ATTENDED_R_P3', 'SYMPOSIUM_ATTENDED_R_P3', 'ADVISORY_BOARD_ATTENDED_I_P3', 'DOCEREE_BANNER_CLICK_I_P3', 'DOXIMITY_FORMULARYONE_CLICK_THROUGH_I_P3', 'EPOCRATES_DOCALERT_DOCALERT_OPENED_I_P3', 'EPOCRATES_MONOGRAPH_MESSAGE_MONOGRAPH_MESSAGE_PRESCRIBING_INFORMATION_CLICKED_I_P3', 'EPOCRATES_SEARCH_SUGGESTIONS_SEARCH_SUGGESTION_HEADLINE_CLICKED_I_P3', 'IMPIRICUS_MOBILE_CLICK_I_P3', 'INVESTIGATOR_MEETING_ATTENDED_I_P3', 'MARKET_ACCESS_FORMULARY_ASSIST_ATTENDED_I_P3', 'MD_CALC_BRANDALERT_CLICK_I_P3', 'MEDSCAPE_BRANDSPOT_RICH_MEDIA_VIDEO_TIME_SPENT_I_P3', 'MEDSCAPE_FFQ_LINK_CLICK_I_P3', 'MEDSCAPE_RECAP_VIDEO_TIME_SPENT_I_P3', 'MEDSCAPE_XCOD_MEDIA_TIME_SPENT_I_P3', 'MJHLIFESCIENCES_3RD_PARTY_EMAIL_OPEN_I_P3', 'NATIVO_BANNER_CLICKS_I_P3', 'PEERDIRECT_FFQ_CLICK_I_P3', 'PRESCRIBER_POINT_CUSTOM_VISIT_I_P3', 'REACHMD_DESKTOP_PARTICIPATION_I_P3', 'REACHMD_EMAIL_EMAIL_OPEN_I_P3', 'SCIENTIFIC_PRESENTATION_ATTENDED_I_P3', 'SCIENTIFIC_ROUND_TABLES_ATTENDED_I_P3', 'SEARCH_BANNER_CLICKED_I_P3', 'SPONSORSHIP_ATTENDED_I_P3', 'SYMPOSIUM_ATTENDED_I_P3', 'ADVISORY_BOARD_ATTENDED_E_P3', 'DOCEREE_BANNER_CLICK_E_P3', 'DOXIMITY_FORMULARYONE_CLICK_THROUGH_E_P3', 'EPOCRATES_DOCALERT_DOCALERT_OPENED_E_P3', 'EPOCRATES_MONOGRAPH_MESSAGE_MONOGRAPH_MESSAGE_PRESCRIBING_INFORMATION_CLICKED_E_P3', 'EPOCRATES_SEARCH_SUGGESTIONS_SEARCH_SUGGESTION_HEADLINE_CLICKED_E_P3', 'IMPIRICUS_MOBILE_CLICK_E_P3', 'INVESTIGATOR_MEETING_ATTENDED_E_P3', 'MARKET_ACCESS_FORMULARY_ASSIST_ATTENDED_E_P3', 'MD_CALC_BRANDALERT_CLICK_E_P3', 'MEDSCAPE_BRANDSPOT_RICH_MEDIA_LINK_CLICK_E_P3', 'MEDSCAPE_FFQ_RESPONSE_E_P3', 'MEDSCAPE_RECAP_PARTICIPATION_E_P3', 'MEDSCAPE_XCOD_LINK_CLICK_E_P3', 'MJHLIFESCIENCES_3RD_PARTY_EMAIL_CLICK_E_P3', 'NATIVO_BANNER_CLICKS_E_P3', 'SCIENTIFIC_PRESENTATION_ATTENDED_E_P3', 'SCIENTIFIC_ROUND_TABLES_ATTENDED_E_P3', 'SEARCH_BANNER_CLICKED_E_P3', 'SPONSORSHIP_ATTENDED_E_P3', 'SYMPOSIUM_ATTENDED_E_P3','ADDITIONAL_DATA_COUNT_P3', 'BRAND_EXPERIENCE_COUNT_P3', 'EFFICACY_COUNT_P3', 'EVENT_COUNT_P3', 'MECHANISM_OF_DISEASE_COUNT_P3', 'MISCELLANEOUS_COUNT_P3', 'PATIENT_ADHERENCE_SUPPORT_COUNT_P3', 'SOCIAL_PROOF_COUNT_P3'    
}

# ============================================================
# Train/test config
# ============================================================

TRAIN_RATIO = 0.80
SEED = 42

# ============================================================
# LightGBM config
# ============================================================

TOP_K = 1000

NUM_ITERATIONS = 250
LEARNING_RATE = 0.02
NUM_LEAVES = 31
# MAX_DEPTH = 5
MIN_DATA_IN_LEAF = 500

FEATURE_FRACTION = 0.8   # colsample_bytree equivalent
BAGGING_FRACTION = 0.8   # subsample equivalent
BAGGING_FREQ = 5

MAX_BIN = 127
BIN_CONSTRUCT_SAMPLE_CNT = 200000

LAMBDA_L1 = 0.1
MIN_GAIN_TO_SPLIT = 0.01

# ============================================================
# Thread config
# ============================================================

NUM_THREADS = psutil.cpu_count(logical=False)

if NUM_THREADS is None:
    NUM_THREADS = 64

NUM_THREADS = min(NUM_THREADS, 64)

# ============================================================
# Data loading config
# ============================================================

ARROW_BATCH_SIZE = 2048

ARRAY_DTYPE = np.float32

# ============================================================
# SHAP config
# ============================================================

SHAP_BATCH_SIZE = 512

# ============================================================
# Quick checks
# ============================================================

print("DATA_PATH:")
print(DATA_PATH)

print("\nOUTPUT_DIR:")
print(OUTPUT_DIR)

print("\nNUM_THREADS:")
print(NUM_THREADS)

In [0]:
print("===================================================")
print("MACHINE CHECK")
print("===================================================")

vm = psutil.virtual_memory()

print(f"CPU logical cores       : {psutil.cpu_count(logical=True)}")
print(f"CPU physical cores      : {psutil.cpu_count(logical=False)}")
print(f"LightGBM NUM_THREADS    : {NUM_THREADS}")
print(f"Total memory GiB        : {vm.total / (1024 ** 3):,.2f}")
print(f"Available memory GiB    : {vm.available / (1024 ** 3):,.2f}")

print("===================================================")

In [0]:
# ============================================================
# Read Parquet dataset
# ============================================================
import pyarrow.dataset as ds

# Full dataset
dataset = ds.dataset(DATA_PATH, format="parquet")

schema = dataset.schema

all_cols = schema.names

if TARGET_COL not in all_cols:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

# ============================================================
# Helper: detect numeric Arrow types
# ============================================================

def is_arrow_numeric_type(dtype):
    return (
        pa.types.is_integer(dtype)
        or pa.types.is_floating(dtype)
        or pa.types.is_decimal(dtype)
        or pa.types.is_boolean(dtype)
    )

# ============================================================
# Select numeric feature columns
# ============================================================

feature_cols = []

for field in schema:
    col_name = field.name
    dtype = field.type

    if col_name in EXCLUDE_COLS:
        continue

    if is_arrow_numeric_type(dtype):
        feature_cols.append(col_name)

n_features = len(feature_cols)

if n_features == 0:
    raise ValueError("No numeric/decimal feature columns found.")

print("Total columns in source dataset :", len(all_cols))
print("Numeric feature columns used    :", n_features)
print("Target column                   :", TARGET_COL)

print("Sample features:")
print(feature_cols[:20])

# ============================================================
# Feature penalty — suppress leaky/dominant features
# ============================================================

PENALIZED_FEATURES = {
    "TOTAL_NBRX_P3": 0.2,    # 20× harder to split
    "TOTAL_TRX_P3": 0.2,      # 10× harder
    "PD_TRX_P3": 0.2,         # 3× harder
}

feature_penalty = [
    PENALIZED_FEATURES.get(col, 1.0)
    for col in feature_cols
]

print("\nPenalized features:")
for col, pen in PENALIZED_FEATURES.items():
    if col in feature_cols:
        print(f"  {col}: penalty = {pen}")
    else:
        print(f"  {col}: NOT in feature_cols (skipped)")

In [0]:
feature_aliases = [f"f_{i}" for i in range(n_features)]

feature_mapping_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols
})

feature_mapping_path = os.path.join(OUTPUT_DIR, "feature_alias_mapping_wo_segment.csv")

feature_mapping_df.to_csv(feature_mapping_path, index=False)

print("Feature mapping saved to:")
print(feature_mapping_path)

feature_mapping_df.head()

In [0]:
# ============================================================
# Count rows
# ============================================================

n_rows = dataset.count_rows()

print("Total rows:", n_rows)
print("Total features:", n_features)

# ============================================================
# Random train/test mask
# ============================================================

rng = np.random.default_rng(SEED)

train_mask_all = rng.random(n_rows) < TRAIN_RATIO

n_train = int(train_mask_all.sum())
n_test = int(n_rows - n_train)

print("Train rows:", n_train)
print("Test rows :", n_test)
print("Train ratio actual:", n_train / n_rows)

In [0]:
print("Allocating train/test arrays...")

start_time = time.time()

X_train = np.empty(
    shape=(n_train, n_features),
    dtype=ARRAY_DTYPE,
    order="C"
)

X_test = np.empty(
    shape=(n_test, n_features),
    dtype=ARRAY_DTYPE,
    order="C"
)

y_train = np.empty(
    shape=(n_train,),
    dtype=ARRAY_DTYPE
)

y_test = np.empty(
    shape=(n_test,),
    dtype=ARRAY_DTYPE
)

elapsed = time.time() - start_time

print(f"Allocation complete in {elapsed:,.2f} seconds")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

In [0]:
# ============================================================
# Columns to scan
# ============================================================

scan_cols = [TARGET_COL] + feature_cols

# Cast every scanned column to float32
cast_schema = pa.schema([
    (col, pa.float32())
    for col in scan_cols
])

# ============================================================
# Scanner
# ============================================================

scanner = dataset.scanner(
    columns=scan_cols,
    batch_size=ARROW_BATCH_SIZE,
    use_threads=True
)

# ============================================================
# Fill arrays batch by batch
# ============================================================

global_row_pos = 0
train_write_pos = 0
test_write_pos = 0

start_time = time.time()

for batch_id, record_batch in enumerate(scanner.to_batches(), start=1):
    batch_n = record_batch.num_rows

    # Convert this batch to Arrow table and cast decimals/numerics to float32
    table = pa.Table.from_batches([record_batch])
    table = table.cast(cast_schema, safe=False)

    # Convert batch to pandas, then NumPy
    # This is temporary and only batch-sized.
    pdf_batch = table.to_pandas(
        split_blocks=True,
        self_destruct=True
    )

    batch_arr = pdf_batch.to_numpy(
        dtype=np.float32,
        copy=True
    )

    # First column is target
    y_batch = batch_arr[:, 0]

    # Replace NULL / NaN / inf target with 0
    bad_y_mask = ~np.isfinite(y_batch)
    if bad_y_mask.any():
        y_batch[bad_y_mask] = 0.0

    # Remaining columns are features
    X_batch = batch_arr[:, 1:]

    # Replace feature inf with NaN.
    # Keep NaN as missing values for LightGBM.
    bad_x_mask = np.isinf(X_batch)
    if bad_x_mask.any():
        X_batch[bad_x_mask] = np.nan

    # Train/test split for this batch
    batch_train_mask = train_mask_all[
        global_row_pos : global_row_pos + batch_n
    ]

    batch_test_mask = ~batch_train_mask

    n_train_batch = int(batch_train_mask.sum())
    n_test_batch = int(batch_test_mask.sum())

    if n_train_batch > 0:
        X_train[
            train_write_pos : train_write_pos + n_train_batch,
            :
        ] = X_batch[batch_train_mask, :]

        y_train[
            train_write_pos : train_write_pos + n_train_batch
        ] = y_batch[batch_train_mask]

        train_write_pos += n_train_batch

    if n_test_batch > 0:
        X_test[
            test_write_pos : test_write_pos + n_test_batch,
            :
        ] = X_batch[batch_test_mask, :]

        y_test[
            test_write_pos : test_write_pos + n_test_batch
        ] = y_batch[batch_test_mask]

        test_write_pos += n_test_batch

    global_row_pos += batch_n

    if batch_id % 25 == 0:
        elapsed = time.time() - start_time
        print(
            f"Batch {batch_id} | rows loaded: {global_row_pos:,} / {n_rows:,} "
            f"| elapsed sec: {elapsed:,.1f}"
        )

    del pdf_batch, batch_arr, X_batch, y_batch, table
    gc.collect()

elapsed = time.time() - start_time

print("===================================================")
print("LOAD COMPLETE")
print("===================================================")
print(f"Rows loaded           : {global_row_pos:,}")
print(f"Train rows written    : {train_write_pos:,}")
print(f"Test rows written     : {test_write_pos:,}")
print(f"Elapsed seconds       : {elapsed:,.2f}")
print("===================================================")

if train_write_pos != n_train:
    raise ValueError(f"Train row mismatch: expected {n_train}, got {train_write_pos}")

if test_write_pos != n_test:
    raise ValueError(f"Test row mismatch: expected {n_test}, got {test_write_pos}")

In [0]:
def summarize_target(y, name):
    total = len(y)
    zero_count = int(np.sum(y == 0))
    nan_count = int(np.sum(~np.isfinite(y)))

    print(f"========== {name} ==========")
    print(f"Rows       : {total:,}")
    print(f"Zero count : {zero_count:,}")
    print(f"NaN/inf    : {nan_count:,}")
    print(f"Min        : {np.nanmin(y):,.6f}")
    print(f"Max        : {np.nanmax(y):,.6f}")
    print(f"Mean       : {np.nanmean(y):,.6f}")
    print()

summarize_target(y_train, "TRAIN TARGET")
summarize_target(y_test, "TEST TARGET")

In [0]:
# ============================================================
# Dataset params
# ============================================================

dataset_params = {
    "max_bin": MAX_BIN,
    "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,
    "feature_pre_filter": False
}

# ============================================================
# LightGBM Dataset objects
# ============================================================

train_data = lgb.Dataset(
    data=X_train,
    label=y_train,
    feature_name=feature_aliases,
    free_raw_data=False,
    params=dataset_params
)

test_data = lgb.Dataset(
    data=X_test,
    label=y_test,
    feature_name=feature_aliases,
    reference=train_data,
    free_raw_data=False,
    params=dataset_params
)

print("LightGBM Dataset objects created.")

In [0]:
# ============================================================
# LightGBM params
# ============================================================

lgb_params = {
    "objective": "poisson",
    "metric": ["poisson", "rmse"],

    "boosting_type": "gbdt",

    "learning_rate": LEARNING_RATE,
    "num_leaves": NUM_LEAVES,
    "min_data_in_leaf": MIN_DATA_IN_LEAF,

    "feature_fraction": FEATURE_FRACTION,   # column sampling
    "bagging_fraction": BAGGING_FRACTION,
    "bagging_freq": BAGGING_FREQ,

    "max_bin": MAX_BIN,
    "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,
    "feature_penalty": feature_penalty,

    "lambda_l1": LAMBDA_L1,
    "min_gain_to_split": MIN_GAIN_TO_SPLIT,

    "force_col_wise": True,
    "num_threads": NUM_THREADS,
    "seed": SEED,
    "verbosity": -1
}
# ============================================================
# Train
# ============================================================

evals_result = {}
start_time = time.time()

booster = lgb.train(
    params=lgb_params,
    train_set=train_data,
    num_boost_round=NUM_ITERATIONS,
    valid_sets=[train_data, test_data],
    valid_names=["train", "test"],
    callbacks=[
        lgb.log_evaluation(period=25),
        lgb.record_evaluation(evals_result),
        lgb.early_stopping(stopping_rounds=30)
    ]
)

elapsed = time.time() - start_time

print("===================================================")
print("TRAINING COMPLETE")
print("===================================================")
print(f"Elapsed seconds: {elapsed:,.2f}")
print("===================================================")

model_path =  "lightgbm_nbrx_model_wo_segment_new_V10.txt"

booster.save_model(model_path)

print("Model saved to:")
print(model_path)

In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(evals_result["train"]["poisson"], label="Train", linewidth=2)
axes[0].plot(evals_result["test"]["poisson"], label="Test", linewidth=2)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Poisson Loss")
axes[0].set_title("Poisson Loss over Iterations")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(evals_result["train"]["rmse"], label="Train", linewidth=2)
axes[1].plot(evals_result["test"]["rmse"], label="Test", linewidth=2)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("RMSE")
axes[1].set_title("RMSE over Iterations")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
start_time = time.time()

pred_train = booster.predict(
    X_train,
    num_threads=NUM_THREADS
)

pred_test = booster.predict(
    X_test,
    num_threads=NUM_THREADS
)

elapsed = time.time() - start_time

print("Prediction complete.")
print(f"Elapsed seconds: {elapsed:,.2f}")

In [0]:
def calculate_metrics(y_true, y_pred, dataset_name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    valid_mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true_valid = y_true[valid_mask]
    y_pred_valid = y_pred[valid_mask]

    r2 = r2_score(y_true_valid, y_pred_valid)

    nonzero_mask = y_true_valid != 0

    if nonzero_mask.sum() > 0:
        mape = (
            np.mean(
                np.abs(
                    (y_true_valid[nonzero_mask] - y_pred_valid[nonzero_mask])
                    / y_true_valid[nonzero_mask]
                )
            )
            * 100.0
        )
    else:
        mape = np.nan

    result = {
        "dataset": dataset_name,
        "r2": float(r2),
        "mape_pct": float(mape) if np.isfinite(mape) else np.nan,
        "rows_total": int(len(y_true)),
        "rows_used_for_r2": int(valid_mask.sum()),
        "rows_used_for_mape": int(nonzero_mask.sum()),
        "zero_target_rows_excluded_from_mape": int((y_true_valid == 0).sum())
    }

    return result


train_metrics = calculate_metrics(y_train, pred_train, "train")
test_metrics = calculate_metrics(y_test, pred_test, "test")

metrics_df = pd.DataFrame([train_metrics, test_metrics])

metrics_path = os.path.join(OUTPUT_DIR, "train_test_metrics_wo_segment_new_V10.csv")
metrics_df.to_csv(metrics_path, index=False)

print(metrics_df)
print()
print("Metrics saved to:")
print(metrics_path)

In [0]:
import pandas as pd

# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)

# Show full column width
pd.set_option('display.max_colwidth', None)

# Show full dataframe width
pd.set_option('display.width', None)

# Example
print(metrics_df)

In [0]:
# ============================================================
# Feature importance
# ============================================================

gain_importance = booster.feature_importance(
    importance_type="gain"
)

split_importance = booster.feature_importance(
    importance_type="split"
)

importance_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols,
    "gain_importance": gain_importance.astype(float),
    "split_importance": split_importance.astype(float)
})

importance_df = importance_df.sort_values(
    by="gain_importance",
    ascending=False
).reset_index(drop=True)

importance_df["gain_rank"] = np.arange(1, len(importance_df) + 1)

importance_path = os.path.join(OUTPUT_DIR, "all_feature_importance_gain_wo_segment_new_V10.csv")
importance_df.to_csv(importance_path, index=False)

print("All feature importance saved to:")
print(importance_path)

importance_df.head(50)

In [0]:
def save_df_to_output(df, filename):
    """
    Save pandas dataframe safely to OUTPUT_DIR.
    """
    local_path = os.path.join(LOCAL_OUTPUT_DIR, filename)

    df.to_csv(local_path, index=False)

    return copy_local_to_output(local_path, filename)


def save_text_list_to_output(values, filename):
    """
    Save list of values safely to OUTPUT_DIR.
    """
    local_path = os.path.join(LOCAL_OUTPUT_DIR, filename)

    with open(local_path, "w") as f:
        for v in values:
            f.write(str(v) + "\n")

    return copy_local_to_output(local_path, filename)


In [0]:
# ============================================================
# Top 1000 features by gain importance
# ============================================================

top_k_effective = min(TOP_K, len(importance_df))

top_features_gain_df = (
    importance_df
    .head(top_k_effective)
    .copy()
    .reset_index(drop=True)
)

top_feature_indices_gain = top_features_gain_df["feature_index"].to_numpy(dtype=np.int32)
top_feature_aliases_gain = top_features_gain_df["feature_alias"].tolist()
top_feature_names_gain = top_features_gain_df["feature"].tolist()

print("Top gain features selected:", len(top_features_gain_df))
print("First 20 gain-based features:")
print(top_feature_names_gain[:20])

pd.DataFrame(top_feature_names_gain).to_csv("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/top_1000_feature_names_gain_only_wo_segment_new_V10.csv")


top_features_gain_df.head(50)

In [0]:
# ============================================================
# SHAP for ALL features on ALL loaded train rows
# ============================================================

print("===================================================")
print("STARTING SHAP FOR ALL FEATURES")
print("===================================================")
print("Train rows:", X_train.shape[0])
print("Features  :", n_features)
print("SHAP batch size:", SHAP_BATCH_SIZE)
print("===================================================")

shap_sum = np.zeros(n_features, dtype=np.float64)
shap_abs_sum = np.zeros(n_features, dtype=np.float64)

total_shap_rows = 0

start_time = time.time()

for start in range(0, X_train.shape[0], SHAP_BATCH_SIZE):
    end = min(start + SHAP_BATCH_SIZE, X_train.shape[0])

    X_batch = X_train[start:end, :]

    # pred_contrib=True returns SHAP values + 1 bias column
    shap_batch_all = booster.predict(
        X_batch,
        pred_contrib=True,
        num_threads=NUM_THREADS
    )

    # Keep only feature SHAP values; drop final bias column
    shap_batch = shap_batch_all[:, :n_features]

    shap_sum += shap_batch.sum(axis=0)
    shap_abs_sum += np.abs(shap_batch).sum(axis=0)

    total_shap_rows += end - start

    if total_shap_rows % (SHAP_BATCH_SIZE * 10) == 0 or end == X_train.shape[0]:
        elapsed = time.time() - start_time

        print(
            f"Processed SHAP rows: {total_shap_rows:,} / {X_train.shape[0]:,} "
            f"| elapsed sec: {elapsed:,.1f}"
        )

    del X_batch, shap_batch_all, shap_batch
    gc.collect()

elapsed = time.time() - start_time

print("===================================================")
print("SHAP COMPLETE")
print("===================================================")
print("Rows used for SHAP:", total_shap_rows)
print(f"Elapsed seconds   : {elapsed:,.2f}")
print("===================================================")

In [0]:
# ============================================================
# SHAP ranking for ALL features
# ============================================================

mean_shap_train = shap_sum / total_shap_rows
mean_abs_shap_train = shap_abs_sum / total_shap_rows

total_mean_abs_shap = float(np.sum(mean_abs_shap_train))

if total_mean_abs_shap > 0:
    percent_impact_shap = 100.0 * mean_abs_shap_train / total_mean_abs_shap
else:
    percent_impact_shap = np.zeros_like(mean_abs_shap_train)

shap_rank_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols,
    "mean_shap_train": mean_shap_train,
    "mean_abs_shap_train": mean_abs_shap_train,
    "percent_impact_shap": percent_impact_shap
})

shap_rank_df = shap_rank_df.sort_values(
    by="mean_abs_shap_train",
    ascending=False
).reset_index(drop=True)

shap_rank_df["shap_rank"] = np.arange(1, len(shap_rank_df) + 1)
shap_rank_df["cumulative_percent_impact_shap"] = shap_rank_df["percent_impact_shap"].cumsum()

# Reorder columns
shap_rank_df = shap_rank_df[
    [
        "shap_rank",
        "feature_index",
        "feature_alias",
        "feature",
        "mean_shap_train",
        "mean_abs_shap_train",
        "percent_impact_shap",
        "cumulative_percent_impact_shap"
    ]
]

shap_rank_df.to_csv(
    "shap_ranking_all_features_all_train_rows_wo_segment_new_V10.csv"
)

print("Top 20 features by SHAP:")
display(shap_rank_df.head(20))
shap_rank_df.shape

In [0]:
# ============================================================
# Top features based on SHAP
# ============================================================

top_1000_shap_df = (
    shap_rank_df
    .head(min(1000, len(shap_rank_df)))
    .copy()
    .reset_index(drop=True)
)

top_100_shap_df = (
    shap_rank_df
    .head(min(100, len(shap_rank_df)))
    .copy()
    .reset_index(drop=True)
)


top_1000_shap_df.to_csv(
    "top_1000_features_by_shap_wo_segment_new_V10.csv"
)

top_100_shap_df.to_csv(
    "top_100_features_by_shap_wo_segment_new_V10.csv"
)


print("Top 1000 SHAP features:", len(top_1000_shap_df))
print("Top 100 SHAP features :", len(top_100_shap_df))

display(top_100_shap_df)

In [0]:
# ============================================================
# Collect row-level SHAP values for top 100 SHAP features
# ============================================================

top100_feature_indices = top_100_shap_df["feature_index"].to_numpy(dtype=np.int32)
top100_feature_names = top_100_shap_df["feature"].tolist()

BEESWARM_ROWS = 100

if BEESWARM_ROWS is None:
    plot_row_indices = np.arange(X_train.shape[0])
else:
    plot_n = min(BEESWARM_ROWS, X_train.shape[0])

    rng = np.random.default_rng(SEED)

    plot_row_indices = rng.choice(
        X_train.shape[0],
        size=plot_n,
        replace=False
    )

plot_row_indices = np.sort(plot_row_indices)

print("Rows used for beeswarm plot:", len(plot_row_indices))
print("Features used for beeswarm plot:", len(top100_feature_indices))


def collect_shap_for_selected_features(
    booster,
    X,
    row_indices,
    feature_indices,
    batch_size,
    num_threads
):
    shap_blocks = []
    feature_value_blocks = []

    total_rows = len(row_indices)
    rows_done = 0

    start_time = time.time()

    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)

        batch_rows = row_indices[start:end]

        X_batch_full = X[batch_rows, :]

        shap_batch_all = booster.predict(
            X_batch_full,
            pred_contrib=True,
            num_threads=num_threads
        )

        shap_batch_selected = shap_batch_all[:, feature_indices]
        X_batch_selected = X_batch_full[:, feature_indices]

        shap_blocks.append(shap_batch_selected.astype(np.float32))
        feature_value_blocks.append(X_batch_selected.astype(np.float32))

        rows_done += end - start

        if rows_done % (batch_size * 10) == 0 or rows_done == total_rows:
            elapsed = time.time() - start_time

            print(
                f"Collected plot SHAP rows: {rows_done:,} / {total_rows:,} "
                f"| elapsed sec: {elapsed:,.1f}"
            )

        del X_batch_full, shap_batch_all, shap_batch_selected, X_batch_selected
        gc.collect()

    shap_selected = np.vstack(shap_blocks)
    X_selected = np.vstack(feature_value_blocks)

    return shap_selected, X_selected


shap_top100_plot, X_top100_plot = collect_shap_for_selected_features(
    booster=booster,
    X=X_train,
    row_indices=plot_row_indices,
    feature_indices=top100_feature_indices,
    batch_size=SHAP_BATCH_SIZE,
    num_threads=NUM_THREADS
)

X_top100_plot_df = pd.DataFrame(
    X_top100_plot,
    columns=top100_feature_names
)

print("SHAP top 100 plot matrix:", shap_top100_plot.shape)
print("X top 100 plot matrix   :", X_top100_plot_df.shape)

In [0]:
#
#  ============================================================
# Beeswarm chart for top 100 SHAP features
# ============================================================
!pip install shap
import shap

import matplotlib.pyplot as plt

plt.figure()

shap.summary_plot(
    shap_top100_plot,
    X_top100_plot_df,
    feature_names=top100_feature_names,
    max_display=100,
    plot_type="dot",
    show=False,
    plot_size=(14, 30)
)

plt.tight_layout()



plt.show()



In [0]:
# ============================================================
# % impact SHAP chart for top 100 features
# ============================================================

impact_plot_df = (
    top_100_shap_df
    .copy()
    .sort_values("percent_impact_shap", ascending=True)
)

plt.figure(figsize=(14, 28))

plt.barh(
    impact_plot_df["feature"],
    impact_plot_df["percent_impact_shap"]
)

plt.xlabel("% impact based on mean(|SHAP value|)")
plt.ylabel("Feature")
plt.title("Top 100 Features by SHAP % Impact")

plt.tight_layout()




plt.show()

print("% impact SHAP chart saved to:")


In [0]:
# ============================================================
# Mean absolute SHAP chart for top 100 features
# ============================================================

mean_abs_plot_df = (
    top_100_shap_df
    .copy()
    .sort_values("mean_abs_shap_train", ascending=True)
)

plt.figure(figsize=(14, 28))

plt.barh(
    mean_abs_plot_df["feature"],
    mean_abs_plot_df["mean_abs_shap_train"]
)

plt.xlabel("mean(|SHAP value|) on train data")
plt.ylabel("Feature")
plt.title("Top 100 Features by Mean Absolute SHAP")

plt.tight_layout()




plt.show()

print("Mean absolute SHAP chart saved to:")


In [0]:
# ============================================================
# Final selected dataframe metadata by SHAP
# ============================================================

selected_features_by_shap = top_1000_shap_df["feature"].tolist()

print("Number of SHAP-selected features:", len(selected_features_by_shap))
print("First 20 SHAP-selected features:")
print(selected_features_by_shap[:20])

final_feature_selection_summary_df = pd.DataFrame({
    "selected_feature_rank": np.arange(1, len(selected_features_by_shap) + 1),
    "feature": selected_features_by_shap
})


final_feature_selection_summary_df.to_csv("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/final_selected_top_1000_features_by_shap_wo_segment_new_V10.csv")

final_feature_selection_summary_df.head(50)